# Unique fasta 
To do the in silico digestion we need to have a fasta file of the full proteome that cdoes not contain duplicated sequences. 

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
from Bio import SeqIO
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/02_In-Silico_Digestion"):
    print("WARNING: The working directory is not set to the '02_In-Silico_Digestion'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_In-Silico_Digestion' folder (\"{cwd}\").")

In [ ]:
# Data directory for fasta data
fasta_dir = "../01_UniProt/data/raw_data/fasta"
duplicates_dir = "../01_UniProt/data/sequence_duplicates"
unique_dir = "../01_UniProt/data/unique"
digestion_input_dir = "data/digestion_input"

## Fasta files

In [ ]:
fasta_canonical = os.path.join(fasta_dir, 'uniprotkb_canonical_2026_03_31.fasta')
fasta_full_proteome = os.path.join(fasta_dir, 'uniprotkb_full_proteome_2026_03_31.fasta')
fasta_alternative_product = os.path.join(fasta_dir, 'uniprotkb_alternative_products_isoforms_2026_03_31.fasta')

fasta_unique_isoform = os.path.join(duplicates_dir, 'unique_isoform_duplicates.fasta')
fasta_unique_canonical = os.path.join(duplicates_dir, 'unique_canonical_duplicates.fasta')

## Duplicates list for filtering

In [ ]:
duplicates_list = pd.read_csv(os.path.join(duplicates_dir, 'duplicates_list_01042026.csv'))

# Filter large fasta file and merge with fasta file of unique identifiers

In [ ]:
duplicates_set = set(duplicates_list["ID"]) #for faster lookup
duplicates_set.add('P0DPK4')
duplicates_set.add('Q96PT3')

fasta_no_duplicates = os.path.join(digestion_input_dir, "fasta_no_duplicates.fasta")

removed_count = 0
kept_count = 0

with open(fasta_no_duplicates, "w") as output_handle:
    for record in SeqIO.parse(fasta_full_proteome, "fasta"):
        # UniProt headers: sp|P12345|NAME_HUMAN -> split by '|'
        # index 0: sp, index 1: P12345, index 2: NAME_HUMAN
        parts = record.id.split('|')
        
        # Extract the accession if the header matches the format
        accession = parts[1] if len(parts) > 1 else record.id
        
        if accession not in duplicates_set:
            SeqIO.write(record, output_handle, "fasta")
            kept_count += 1
        else:
            removed_count += 1

print(f"Removed: {removed_count} | Kept: {kept_count}")

In [ ]:
# append unique canonical fasta to filtered fasta
with open(fasta_unique_canonical, "r") as src, open(fasta_no_duplicates, "a") as dest:
    # Ensure there's a newline before starting to append
    #dest.write("\n") 
    dest.write(src.read())

In [ ]:
# append unique isoform fasta to filtered fasta
with open(fasta_unique_isoform, "r") as src, open(fasta_no_duplicates, "a") as dest:
    # Ensure there's a newline before starting to append
    #dest.write("\n") 
    dest.write(src.read())

In [ ]:
# let's have a look at the final fasta file
fasta_unique = read_fastafile(os.path.join(digestion_input_dir, "fasta_no_duplicates.fasta"))
fasta_unique